# 词性标注 - 初步步骤：处理文本文件、创建词汇表以及处理未知单词

在本节讲座笔记中，你将从一个带标注的数据集中创建词汇表，并学习在处理其他文本来源时如何处理该词汇表中未出现的单词。除此之外，你还将学习如何：

- 读取文本文件
- 使用 defaultdict
- 处理字符串数据

In [2]:
import string
from collections import defaultdict

### Read Text Data

文件 `WSJ_02-21.pos` 中提供了一个取自《华尔街日报》的带标注数据集。

要读取此文件，你可以使用 Python 的上下文管理器，方法是使用 `with` 关键字并指定你要读取的文件名。要实际将文件内容保存到内存中，你需要使用 `readlines()` 方法并将其返回值存储在一个变量中。

Python 的上下文管理器非常出色，因为你不需要显式关闭与文件的连接，这会在后台自动完成：

In [1]:
# Read lines from 'WSJ_02-21.pos' file and save them into the 'lines' variable
with open("WSJ_02-21.pos", 'r') as f:
    lines = f.readlines()

To check the contents of the dataset you can print the first 5 lines:

In [3]:
# Print columns for reference
print("\t\tWord", "\tTag\n")

# Print first five lines of the dataset
for i in range(5):
    print(f'line number {i+1}: {lines[i]}')

		Word 	Tag

line number 1: In	IN

line number 2: an	DT

line number 3: Oct.	NNP

line number 4: 19	CD

line number 5: review	NN



数据集中的每一行包含一个单词及其对应的词性标签。然而，由于打印是使用格式化字符串完成的，因此可以推断出 **单词** 和 **标签** 之间由制表符（或一些空格）分隔，并且每行末尾有一个换行符（注意每行之间有一个空格）。

如果你想了解这些标签的含义，可以查看 [这里](https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html)。

为了更好地理解数据集中的信息结构，建议打印其未格式化的版本：

<URL_content>
[URL 标题] 宾夕法尼亚州立大学树库项目使用的词性标签字母顺序列表：
[URL 链接] https://www.ling.upenn.edu/courses/Fall_2003/ling001/penn_treebank_pos.html
[URL 内容开始]
### 宾夕法尼亚州立大学树库项目使用的词性标签字母顺序列表：

编号 | 标签 | 描述
---|---|---
 1.  | CC  | 并列连词
 2.  | CD  | 基数词
 3.  | DT  | 限定词
 4.  | EX  | 存在型 there
 5.  | FW  | 外来词
 6.  | IN  | 介词或从属连词
 7.  | JJ  | 形容词
 8.  | JJR  | 形容词，比较级
 9.  | JJS  | 形容词，最高级
 10.  | LS  | 列表项标记
 11.  | MD  | 情态动词
 12.  | NN  | 名词，单数或不可数
 13.  | NNS  | 名词，复数
 14.  | NNP  | 专有名词，单数
 15.  | NNPS  | 专有名词，复数
 16.  | PDT  | 前限定词
 17.  | POS  | 所有格结尾
 18.  | PRP  | 人称代词
 19.  | PRP$  | 所有格代词
 20.  | RB  | 副词
 21.  | RBR  | 副词，比较级
 22.  | RBS  | 副词，最高级
 23.  | RP  | 小品词
 24.  | SYM  | 符号
 25.  | TO  | to
 26.  | UH  | 感叹词
 27.  | VB  | 动词，原形
 28.  | VBD  | 动词，过去时
 29.  | VBG  | 动词，动名词或现在分词
 30.  | VBN  | 动词，过去分词
 31.  | VBP  | 动词，非第三人称单数现在时
 32.  | VBZ  | 动词，第三人称单数现在时
 33.  | WDT  | Wh-限定词
 34.  | WP  | Wh-代词
 35.  | WP$  | 所有格 Wh-代词
 36.  | WRB  | Wh-副词
[URL 内容结束]
</URL_content>

In [4]:
# Print first line (unformatted)
lines[0]

'In\tIN\n'

确实，单词和标签之间有一个制表符，并且每行末尾有一个换行符。

### 创建词汇表

现在你已经了解了数据集的结构，你将基于它创建一个词汇表。词汇表由数据集中至少出现 2 次的所有单词组成。
为此，请按照以下步骤操作：
- 仅从数据集中获取单词
- 使用 defaultdict 统计每个单词出现的次数
- 过滤字典，仅保留出现至少 2 次的单词
- 从过滤后的字典中创建一个列表
- 对列表进行排序

对于步骤 1，你可以利用以下事实：每个单词和标签由制表符分隔，并且单词总是排在第一位。使用列表推导式，可以像这样创建单词列表：

In [13]:
# Get the words from each line in the dataset
words = [line.split('\t')[0] for line in lines]

步骤 2 可以通过利用 `defaultdict` 轻松完成。如果你不熟悉 defaultdict，它们是一种特殊的字典，**当你尝试访问一个不存在的键时，会返回该类型的“零”值**。既然你想要的是单词的频率，你应该使用 `int` 类型来定义 defaultdict。

现在你不需要担心单词不在字典中的情况，因为获取该键的值只会返回零。是不是很酷？

In [14]:
# Define defaultdict of type 'int'
freq = defaultdict(int)

# Count frequency of ocurrence for each word in the dataset
for word in words:
    freq[word] += 1

Filtering the `freq` dictionary can be done using list comprehensions again (aren't they handy?). You should filter out words that appeared only once and also words that are just a newline character:

In [15]:
# Create the vocabulary by filtering the 'freq' dictionary
vocab = [k for k, v in freq.items() if (v > 1 and k != '\n')]

Finally, the `sort` method will take care of the final step. Notice that it changes the list directly so you don't need to reassign the `vocab` variable:

In [16]:
# Sort the vocabulary
vocab.sort()

# Print some random values of the vocabulary
for i in range(4000, 4005):
    print(vocab[i])

Early
Earnings
Earth
Earthquake
East


Now you have successfully created a vocabulary from the dataset. **Great job!** The vocabulary is quite extensive so it is not printed out but you can still do so by creating a cell and running something like `print(vocab)`. 

At this point you will usually write the vocabulary into a file for future use, but that is out of the scope of this notebook. If you are curious it is very similar to how you read the file at the beginning of this notebook.


## Processing new text sources

### Dealing with unknown words

Now that you have a vocabulary, you will use it when processing new text sources. **A new text will have words that do not appear in the current vocabulary**. To tackle this, you can simply classify each new word as an unknown one, but you can do better by creating a function that tries to classify the type of each unknown word and assign it a corresponding `unknown token`. 

This function will do the following checks and return an appropriate token:

   - Check if the unknown word contains any character that is a digit 
       - return `--unk_digit--`
   - Check if the unknown word contains any punctuation character 
       - return `--unk_punct--`
   - Check if the unknown word contains any upper-case character 
       - return `--unk_upper--`
   - Check if the unknown word ends with a suffix that could indicate it is a noun, verb, adjective or adverb 
        - return `--unk_noun--`, `--unk_verb--`, `--unk_adj--`, `--unk_adv--` respectively

If a word fails to fall under any condition then its token will be a plain `--unk--`. The conditions will be evaluated in the same order as listed here. So if a word contains a punctuation character but does not contain digits, it will fall under the second condition. To achieve this behaviour some if/elif statements can be used along with early returns. 

This function is implemented next. Notice that the `any()` function is being heavily used. It returns `True` if at least one of the cases it evaluates is `True`.

In [17]:
def assign_unk(word):
    """
    Assign tokens to unknown words
    """
    
    # Punctuation characters
    # Try printing them out in a new cell!
    punct = set(string.punctuation)
    
    # Suffixes
    noun_suffix = ["action", "age", "ance", "cy", "dom", "ee", "ence", "er", "hood", "ion", "ism", "ist", "ity", "ling", "ment", "ness", "or", "ry", "scape", "ship", "ty"]
    verb_suffix = ["ate", "ify", "ise", "ize"]
    adj_suffix = ["able", "ese", "ful", "i", "ian", "ible", "ic", "ish", "ive", "less", "ly", "ous"]
    adv_suffix = ["ward", "wards", "wise"]

    # Loop the characters in the word, check if any is a digit
    if any(char.isdigit() for char in word):
        return "--unk_digit--"

    # Loop the characters in the word, check if any is a punctuation character
    elif any(char in punct for char in word):
        return "--unk_punct--"

    # Loop the characters in the word, check if any is an upper case character
    elif any(char.isupper() for char in word):
        return "--unk_upper--"

    # Check if word ends with any noun suffix
    elif any(word.endswith(suffix) for suffix in noun_suffix):
        return "--unk_noun--"

    # Check if word ends with any verb suffix
    elif any(word.endswith(suffix) for suffix in verb_suffix):
        return "--unk_verb--"

    # Check if word ends with any adjective suffix
    elif any(word.endswith(suffix) for suffix in adj_suffix):
        return "--unk_adj--"

    # Check if word ends with any adverb suffix
    elif any(word.endswith(suffix) for suffix in adv_suffix):
        return "--unk_adv--"
    
    # If none of the previous criteria is met, return plain unknown
    return "--unk--"


A POS tagger will always encounter words that are not within the vocabulary that is being used. By augmenting the dataset to include these `unknown word tokens` you are helping the tagger to have a better idea of the appropriate tag for these words. 

### Getting the correct tag for a word

All that is left is to implement a function that will get the correct tag for a particular word taking special considerations for unknown words. Since the dataset provides each word and tag within the same line and a word being known depends on the vocabulary used, these two elements should be arguments to this function.

This function should check if a line is empty and if so, it should return a placeholder word and tag, `--n--` and `--s--` respectively. 

If not, it should process the line to return the correct word and tag pair, considering if a word is unknown in which scenario the function `assign_unk()` should be used.

The function is implemented next. Notice That the `split()` method can be used without specifying the delimiter, in which case it will default to any whitespace.

In [18]:
def get_word_tag(line, vocab):
    # If line is empty return placeholders for word and tag
    if not line.split():
        word = "--n--"
        tag = "--s--"
    else:
        # Split line to separate word and tag
        word, tag = line.split()
        # Check if word is not in vocabulary
        if word not in vocab: 
            # Handle unknown word
            word = assign_unk(word)
    return word, tag

Now you can try this function with some examples to test that it is working as intended:

In [19]:
get_word_tag('\n', vocab)

('--n--', '--s--')

Since this line only includes a newline character it returns a placeholder word and tag.

In [20]:
get_word_tag('In\tIN\n', vocab)

('In', 'IN')

This one is a valid line and the function does a fair job at returning the correct (word, tag) pair.

In [21]:
get_word_tag('tardigrade\tNN\n', vocab)

('--unk--', 'NN')

This line includes a noun that is not present in the vocabulary. 

The `assign_unk` function fails to detect that it is a noun so it returns an `unknown token`.

In [22]:
get_word_tag('scrutinize\tVB\n', vocab)

('--unk_verb--', 'VB')

This line includes a verb that is not present in the vocabulary. 

In this case the `assign_unk` is able to detect that it is a verb so it returns an `unknown verb token`.

**Congratulations on finishing this lecture notebook!** Now you should be more familiar with working with text data and have a better understanding of how a basic POS tagger works.

**Keep it up!**